# 数据处理

## 读取数据
- 建立数据清单
- 训练集和验证集都是`list`，元素是`tuple`，存放了`(Path("*"), label)`
- 测试集只有数据没有标签

In [ ]:
from pathlib import Path

DATA_ROOT = Path(
    r"E:\projects\PyCharmProject\rookie\machine_learning\CV\ResNet"
    r"\Tiny_Images\tiny-imagenet-200"
)

TRAIN_ROOT = DATA_ROOT / "train"
VAL_ROOT = DATA_ROOT / "val"
TEST_ROOT = DATA_ROOT / "test"

- 用 `winds.txt` 是训练集的映射
- `class_to_idx`是train文件夹下每个类别标签和图片集的映射

In [ ]:
# 1. 以 wnids.txt 的顺序建立唯一、固定的类别映射
wnids = []
text = (DATA_ROOT / "wnids.txt").read_text(encoding="utf-8")
for line in text.splitlines():
    stripped = line.strip()
    if stripped:              # 非空才加入
        wnids.append(stripped)

class_to_idx = {wnid: idx for idx, wnid in enumerate(wnids)}

In [102]:
len(class_to_idx)

200

- 训练集清单

In [ ]:
# 2. 训练集清单：
# train/<wnid>/images/xxx.JPEG
# p.parent       = images
# p.parent.parent = wnid
# 现在train_sample里面的数据是(path, label)，共100,000个数据（200*500）
train_samples = []
for path in sorted(TRAIN_ROOT.glob("*/*/*.JPEG")):
    wnid = path.parent.parent.name
    label = class_to_idx[wnid]
    train_samples.append((path.relative_to(DATA_ROOT), label))
train_samples

- 验证集清单

In [58]:
# 3. 验证集清单：先读取文件名到 WNID 的映射
val_annotation = {}
annotation_path = VAL_ROOT / "val_annotations.txt"

for line in annotation_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue

    fields = line.split("\t")
    image_name, wnid = fields[0], fields[1]
    val_annotation[image_name] = wnid

val_paths = sorted((VAL_ROOT / "images").glob("*.JPEG"))

In [59]:
# 防止标注缺项或存在无标注文件
val_file_names = {path.name for path in val_paths}

val_samples = []
for path in val_paths:
    wnid = val_annotation[path.name]
    label = class_to_idx[wnid]
    val_samples.append((path.relative_to(DATA_ROOT), label))
val_samples

[(WindowsPath('val/images/val_0.JPEG'), 163),
 (WindowsPath('val/images/val_1.JPEG'), 1),
 (WindowsPath('val/images/val_10.JPEG'), 136),
 (WindowsPath('val/images/val_100.JPEG'), 123),
 (WindowsPath('val/images/val_1000.JPEG'), 92),
 (WindowsPath('val/images/val_1001.JPEG'), 65),
 (WindowsPath('val/images/val_1002.JPEG'), 45),
 (WindowsPath('val/images/val_1003.JPEG'), 25),
 (WindowsPath('val/images/val_1004.JPEG'), 87),
 (WindowsPath('val/images/val_1005.JPEG'), 171),
 (WindowsPath('val/images/val_1006.JPEG'), 38),
 (WindowsPath('val/images/val_1007.JPEG'), 163),
 (WindowsPath('val/images/val_1008.JPEG'), 163),
 (WindowsPath('val/images/val_1009.JPEG'), 67),
 (WindowsPath('val/images/val_101.JPEG'), 51),
 (WindowsPath('val/images/val_1010.JPEG'), 158),
 (WindowsPath('val/images/val_1011.JPEG'), 116),
 (WindowsPath('val/images/val_1012.JPEG'), 122),
 (WindowsPath('val/images/val_1013.JPEG'), 155),
 (WindowsPath('val/images/val_1014.JPEG'), 7),
 (WindowsPath('val/images/val_1015.JPEG'),

- 测试集清单

In [95]:
# 4. 测试集没有标签
test_paths = [
    path.relative_to(DATA_ROOT)
    for path in sorted((TEST_ROOT / "images").glob("*.JPEG"))
]

## 处理数据

### Dataset 与 DataLoader

下面的代码只负责把样本清单转换成可训练的 Tensor，不包含模型和训练循环。

数据形状必须始终满足：

```text
单张 JPEG（最终统一为 RGB）    [64, 64, 3]   # PIL 的通道在后
ToTensor 之后                  [3, 64, 64]   # PyTorch 的通道在前
DataLoader 组 Batch 之后       [B, 3, 64, 64]
标签 Batch                     [B]
```

In [62]:
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


# 当前数据的输入尺寸。ResNet 的 stem 设计也必须以这个尺寸为前提。
IMAGE_SIZE = 64

# 训练 Batch 可以稍小，给模型和显存留下空间。
TRAIN_BATCH_SIZE = 128

# 验证和测试没有反向传播，不需要保存梯度，可以使用更大的 Batch。
EVAL_BATCH_SIZE = 256

# Windows 下的 Jupyter 在多进程中序列化 notebook 中定义的 Dataset 时容易出问题。
# 因此先使用 0，保证接口稳定；在独立 .py 训练脚本中通常可以使用 4 或 8。
NUM_WORKERS = 0


# 这组统计量是对当前 train 清单中的 100,000 张图执行以下步骤后得到的：
# convert("RGB") -> [0, 1] 浮点值 -> 按通道流式累计 sum 与 sum_square。
# 验证集和测试集没有参与统计，避免数据泄漏；图像增强也没有参与统计。
#
# 每个通道使用：
#   mean = E[x]
#   std  = sqrt(E[x^2] - E[x]^2)
TRAIN_MEAN = (0.48023694, 0.44806704, 0.39750364)
TRAIN_STD = (0.27643643, 0.26886328, 0.28158993)

- Transformer
> 1. 随机裁剪（先填充，再裁剪），这样可以改变图片位置
> 2. 随机翻转（只有训练集才做）
> 3. 将image 转成 Tensor
> 4. 标准化

In [97]:
train_transform = transforms.Compose([
    transforms.RandomCrop(
        size=IMAGE_SIZE,
        padding=4,
        padding_mode="reflect",
    ), # 随机裁剪——先填充，再随机才回去
    transforms.RandomHorizontalFlip(p=0.5), # 0.5 概率左右翻转
    transforms.ToTensor(), # 维度重排——HWC → CHW
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD), # 标准化
])


# 验证集和测试集必须可复现，不能使用随机裁剪或随机翻转。
# 形状：[64, 64, 3] -> [3, 64, 64]，数值随后进行相同的逐通道标准化。
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD),
])

print(train_transform)
print(eval_transform)

Compose(
    RandomCrop(size=(64, 64), padding=4)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=(0.48023694, 0.44806704, 0.39750364), std=(0.27643643, 0.26886328, 0.28158993))
)
Compose(
    ToTensor()
    Normalize(mean=(0.48023694, 0.44806704, 0.39750364), std=(0.27643643, 0.26886328, 0.28158993))
)


- 自定义Dataset

In [92]:
class ImageClassificationDataset(Dataset):
    """把 (相对路径, 整数标签) 清单映射成 (Tensor, 标量标签)。"""

    def __init__(self, samples, data_root, transform):
        # samples 形如：
        # [(Path("train/wnid/images/x.JPEG"), 0), ...]
        # 这里只保存路径和标签，不提前把图片像素放入内存。
        self.samples = samples
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        relative_path, label = self.samples[index]
        image_path = self.data_root / relative_path

        if not image_path.is_file():
            raise FileNotFoundError(f"找不到图片：{image_path}")

        with Image.open(image_path) as image:
            # 数据集里同时存在 RGB 和 L（灰度）模式。
            # 必须先转成 RGB，保证每个样本最终都是 [3, 64, 64]。
            image = image.convert("RGB")

            # transform 输出：[3, 64, 64]，dtype=torch.float32。
            image = self.transform(image)

        # CrossEntropyLoss 默认要求 target 为 int64，形状由 DataLoader 变成 [B]。
        target = torch.tensor(label, dtype=torch.long)
        return image, target


In [93]:
class ImageTestDataset(Dataset):
    """测试集没有标签，因此返回 (图像 Tensor, 相对路径)。"""

    def __init__(self, paths, data_root, transform):
        self.paths = paths
        self.data_root = data_root
        self.transform = transform

        if len(self.paths) == 0:
            raise ValueError("paths 不能为空。")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        relative_path = self.paths[index]
        image_path = self.data_root / relative_path

        if not image_path.is_file():
            raise FileNotFoundError(f"找不到图片：{image_path}")

        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        # 返回字符串而不是 Path，默认 collate_fn 可以直接把路径组成 Python list。
        return image, str(relative_path)

- 建立DataLoader

In [98]:
def build_dataloaders(
    train_samples,
    val_samples,
    test_paths,
    *,
    data_root=DATA_ROOT,
    train_transform=train_transform,
    eval_transform=eval_transform,
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
):
    """根据三份清单一次性创建 train/val/test DataLoader。"""

    train_dataset = ImageClassificationDataset(
        samples=train_samples,
        data_root=data_root,
        transform=train_transform,
    )
    val_dataset = ImageClassificationDataset(
        samples=val_samples,
        data_root=data_root,
        transform=eval_transform,
    )
    test_dataset = ImageTestDataset(
        paths=test_paths,
        data_root=data_root,
        transform=eval_transform,
    )

    # 只有训练集需要 shuffle。验证集和测试集保持固定顺序，
    # 这样预测结果才能和路径、标签稳定对应。
    common_loader_args = {
        "num_workers": num_workers, # 用几个子进程加载数据
        "pin_memory": torch.cuda.is_available(), # 用锁页内存加速主机→显存拷贝
        "persistent_workers": num_workers > 0, # 让 worker 进程跨 epoch 存活
    }

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        drop_last=False,
        **common_loader_args,
    )
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        **common_loader_args,
    )

    return train_loader, val_loader, test_loader


In [99]:
train_loader, val_loader, test_loader = build_dataloaders(
    train_samples=train_samples,
    val_samples=val_samples,
    test_paths=test_paths,
)

print(f"train: {len(train_loader.dataset):,} samples, {len(train_loader)} batches")
print(f"val:   {len(val_loader.dataset):,} samples, {len(val_loader)} batches")
print(f"test:  {len(test_loader.dataset):,} samples, {len(test_loader)} batches")

train: 100,000 samples, 782 batches
val:   10,000 samples, 40 batches
test:  10,000 samples, 40 batches


In [123]:
type(train_loader)
idx = 0
labels = []
for data in enumerate(train_loader):
    print(type(data))

    input, label = data[1], data[0]
    print(type(input))
    print(type(label))
    print(label, len(input))
    input_data, label_data = input[0], input[1]
    print(type(input_data))
    print(type(label_data))
    print(input_data.shape, label_data.shape)
    print(sorted(label_data))
    for label in label_data:
        labels.append(label.item())
    idx += 1
    if idx > 5:
        break

<class 'tuple'>
<class 'list'>
<class 'int'>
0 2
<class 'torch.Tensor'>
<class 'torch.Tensor'>
torch.Size([128, 3, 64, 64]) torch.Size([128])
[tensor(0), tensor(8), tensor(9), tensor(10), tensor(10), tensor(10), tensor(14), tensor(14), tensor(15), tensor(15), tensor(18), tensor(20), tensor(21), tensor(21), tensor(23), tensor(24), tensor(25), tensor(27), tensor(27), tensor(28), tensor(29), tensor(32), tensor(36), tensor(37), tensor(39), tensor(39), tensor(40), tensor(42), tensor(43), tensor(45), tensor(45), tensor(45), tensor(49), tensor(52), tensor(52), tensor(53), tensor(54), tensor(58), tensor(61), tensor(61), tensor(62), tensor(62), tensor(65), tensor(65), tensor(65), tensor(65), tensor(67), tensor(68), tensor(69), tensor(70), tensor(75), tensor(75), tensor(77), tensor(79), tensor(80), tensor(80), tensor(81), tensor(81), tensor(82), tensor(83), tensor(84), tensor(85), tensor(85), tensor(87), tensor(89), tensor(90), tensor(91), tensor(91), tensor(92), tensor(94), tensor(95), tensor(9

In [125]:
sorted(set(labels))

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 13,
 14,
 15,
 16,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 188,
 189,
 190,
 191,
 192,
